# Which features matter for survival on the Titanic?

In [ ]:
from titanic.data import combine_train_test, load_data
from titanic.plotting import plot_count_pairs, plot_distribution_pairs, save_fig

train_df, test_df = load_data()
all_df = combine_train_test(train_df, test_df)

## Demographic features: sex, age and family size

### Sex

In [ ]:
plot_count_pairs(train_df, "Sex", hue="Survived", save_as="survival_by_sex")

Out of roughly 580 men, only about 110 survived (around 19%), while about 230 of roughly 310 women survived (around 74%). Even though there were almost twice as many men as women on board, more than twice as many women survived. This is the largest difference of any feature, so Sex is kept as one of the most important predictors.

### Age

In [ ]:
plot_distribution_pairs(train_df, "Age", hue="Survived", save_as="age_by_survival")

Young children were the group most likely to survive: among children under 5, about 30 survived compared with about 13 who died. Between roughly 15 and 35, where most passengers were, deaths clearly outnumbered survivals, and almost no passengers over 65 survived. Since survival changes with age, especially for young children, Age is kept as a predictor. However, Age is missing for 177 passengers (about 20%), so these values need to be filled before modelling.

### Family size

In [ ]:
train_df["FamilySize"] = train_df["SibSp"] + train_df["Parch"] + 1
plot_count_pairs(train_df, "FamilySize", hue="Survived", save_as="survival_by_family_size")

Passengers travelling alone were the largest group, and most of them did not survive: about 160 survived out of roughly 540 (around 30%). For small families of 2 to 4 people, survivors outnumbered those who died, with family size 4 doing best (about 21 survived compared with 8 who died). For families of 5 or more, very few survived. Since survival depends on family size in this clear way, SibSp and Parch are replaced by the combined FamilySize feature, which could also be grouped into alone, small (2 to 4) and large (5 or more).

## Socio-economic features: class, fare, port and title

These plots show survival rates from the training data, with passenger numbers (n) on each bar.

In [ ]:
from titanic.features import (
    add_family_size,
    add_name_parts,
    add_titles,
    map_sex_to_numeric,
)
from titanic.plotting import plot_correlation_heatmap, plot_survival_rate

socio_df = add_titles(add_name_parts(train_df))

In [ ]:
plot_survival_rate(
    socio_df,
    "Pclass",
    "Class matters within each sex, but a third-class woman\n"
    "still has better odds than a first-class man",
    hue="Sex",
    save_as="survival_by_class_and_sex",
)

**Keep `Pclass`.** Survival falls with class for both sexes: from 97% to 50% for women and from 37% to 14% for men.

In [ ]:
plot_survival_rate(
    socio_df,
    "Titles",
    "Title separates boys from men, which Sex alone cannot do",
    save_as="survival_by_title",
)

**Keep `Titles`.** Among males, Master (boys aged 12 or under where age is known) survived at 57%, against 16% for Mr, which `Sex` alone cannot show.

In [ ]:
plot_survival_rate(
    socio_df,
    "Pclass",
    "Port still matters within third class,\nbut much less than class itself",
    hue="Embarked",
    save_as="survival_by_class_and_port",
)

**`Embarked` is a weak candidate.** In third class, C and Q passengers survived at 38%, against 19% from S, but the gap is smaller in first class and some Q bars have under 5 passengers. Keep it only if it improves validation results.

In [ ]:
heatmap_df = map_sex_to_numeric(add_family_size(socio_df))
plot_correlation_heatmap(
    heatmap_df,
    ["Survived", "Pclass", "Fare", "Sex", "Age", "Family Size"],
    "Fare overlaps with class, so it adds less\nthan its own correlation suggests",
    save_as="correlation_heatmap",
)

**Leave out `Fare` at first.** It is linked to survival (0.26) but also to `Pclass` (-0.55), which has the stronger link (-0.34), so class carries much of its signal. Correlation only measures straight-line links, so this still needs testing.

## Conclusion: which features to keep

We would start with `Sex`, `Pclass`, `Titles`, `Age` (with missing values filled) and family size (replacing `SibSp` and `Parch`). `Embarked` and `Fare` would be tested afterwards and kept only if they improve validation results.

`Age` and family size have correlations close to zero in the heatmap, but this does not conflict with keeping them: young children and small families did best, a curved pattern that a straight-line correlation misses. These plots show associations, not causes, so the final feature set should be confirmed on validation data.